# 1 - CNN Training and Inference
Training and makeing inference in a double digit image ...

## 01: Dependencies and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:

# ==========================================
# PASO 1: CARGAR Y PREPARAR LOS DATOS
# ==========================================
print("Cargando los datos generados...")

# Cargamos los archivos .pt que creamos en el notebook anterior
train_data = torch.load('./data/MNIST_2DIGITS/train_2digits.pt')
test_data = torch.load('./data/MNIST_2DIGITS/test_2digits.pt')

X_train, y_train = train_data['imagenes'], train_data['etiquetas']
X_test, y_test = test_data['imagenes'], test_data['etiquetas']

# TensorDataset agrupa las imágenes con sus etiquetas
# DataLoader nos permite pasar los datos a la red en "lotes" (batches) de 64 en 64
# Esto hace que el entrenamiento sea mucho más rápido y estable
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Datos listos. Lotes de entrenamiento: {len(train_loader)}")

In [ ]:
# ==========================================
# PASO 2: DEFINIR LA ARQUITECTURA DEL MODELO
# ==========================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Primera capa convolucional: entra 1 canal (blanco y negro), salen 16 filtros
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        # Segunda capa convolucional: entran 16, salen 32 filtros
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # Capa para reducir el tamaño de la imagen a la mitad
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Función de activación no lineal
        self.relu = nn.ReLU()
        
        # Después de dos MaxPool, nuestra imagen de 28x56 se reduce a 7x14.
        # Multiplicamos por los 32 filtros resultantes: 32 * 7 * 14 = 3136 características
        self.fc1 = nn.Linear(in_features=3136, out_features=128)
        
        # Capa de salida: 100 neuronas (una para cada número del 00 al 99)
        self.fc2 = nn.Linear(in_features=128, out_features=100)

    def forward(self, x):
        # Pasamos la imagen por las capas
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        
        # Aplanamos los datos para que puedan entrar en la capa lineal
        x = x.view(-1, 3136) 
        
        x = self.relu(self.fc1(x))
        x = self.fc2(x) # La salida final (sin activación porque usaremos CrossEntropyLoss)
        return x

In [ ]:
# ==========================================
# PASO 3: CONFIGURAR EL ENTRENAMIENTO
# ==========================================
# Instanciamos el modelo
modelo = SimpleCNN()

# Función de pérdida estándar para clasificación multiclase
criterion = nn.CrossEntropyLoss()
# Optimizador Adam (aprende rápido y ajusta bien los pesos)
optimizer = optim.Adam(modelo.parameters(), lr=0.001)

In [ ]:
# ==========================================
# PASO 4: BUCLE DE ENTRENAMIENTO
# ==========================================
epocas = 5 # 5 pasadas completas por el dataset son suficientes para este problema sencillo

print("\nIniciando el entrenamiento...")
for epoch in range(epocas):
    modelo.train() # Ponemos el modelo en modo entrenamiento
    perdida_acumulada = 0.0
    
    for inputs, labels in train_loader:
        # 1. Reiniciar los gradientes
        optimizer.zero_grad()
        # 2. Pasar los datos por el modelo (Forward pass)
        outputs = modelo(inputs)
        # 3. Calcular el error (Loss)
        loss = criterion(outputs, labels)
        # 4. Calcular los gradientes (Backward pass)
        loss.backward()
        # 5. Actualizar los pesos
        optimizer.step()
        
        perdida_acumulada += loss.item()
        
    print(f"Época {epoch+1}/{epocas} completada. Pérdida media: {perdida_acumulada/len(train_loader):.4f}")

print("¡Entrenamiento finalizado!")

Cargando los datos generados...
Datos listos. Lotes de entrenamiento: 938

Iniciando el entrenamiento...
Época 1/5 completada. Pérdida media: 0.8543
Época 2/5 completada. Pérdida media: 0.1785
Época 3/5 completada. Pérdida media: 0.1127
Época 4/5 completada. Pérdida media: 0.0808
Época 5/5 completada. Pérdida media: 0.0625
¡Entrenamiento finalizado!


# 2 - CNN Training and Inference
Training and evaluating a CNN model in order to recognize double digit images

In [ ]:
# ==========================================
# PASO 1: INFERENCIA SOBRE EL CONJUNTO DE TEST
# ==========================================
# Ponemos el modelo en modo "evaluación". Esto desactiva ciertas capas 
# (como Dropout si la tuviéramos) que solo se usan en entrenamiento.
modelo.eval()

predicciones_totales = []
etiquetas_reales = []

print("Realizando inferencia sobre los datos de test...")

# torch.no_grad() apaga el cálculo de gradientes.
# Esto hace que la inferencia sea mucho más rápida y consuma menos memoria.
with torch.no_grad():
    for inputs, labels in test_loader:
        # 1. Pasamos las imágenes por el modelo
        outputs = modelo(inputs)
        
        # 2. outputs contiene 100 "puntuaciones" por imagen. 
        # torch.max devuelve el valor máximo y su posición (el número predicho)
        _, predicciones = torch.max(outputs, 1)
        
        # 3. Guardamos las predicciones y las etiquetas reales en listas
        predicciones_totales.extend(predicciones.numpy())
        etiquetas_reales.extend(labels.numpy())

In [ ]:
# ==========================================
# PASO 2: CÁLCULO DE LA MÉTRICA (ACCURACY)
# ==========================================
# Comparamos las listas usando scikit-learn
accuracy = accuracy_score(etiquetas_reales, predicciones_totales)
print(f"\nResultados Finales:")
print(f"Total de imágenes evaluadas: {len(etiquetas_reales)}")
print(f"Accuracy del modelo: {accuracy * 100:.2f}%")

In [ ]:
# ==========================================
# PASO 3: VISUALIZACIÓN DE RESULTADOS
# ==========================================
# El enunciado sugiere incluir matrices de confusión. 
# Como tenemos 100 clases (00 al 99), una matriz completa será gigante, 
# pero es una visualización técnica muy potente para el informe.

matriz = confusion_matrix(etiquetas_reales, predicciones_totales)

plt.figure(figsize=(20, 16))
# Usamos seaborn para dibujar la matriz de calor (heatmap)
sns.heatmap(matriz, cmap="Blues", cbar=False)
plt.title("Matriz de Confusión: Reconocimiento de Números (00-99)", fontsize=18)
plt.xlabel("Predicción del Modelo", fontsize=14)
plt.ylabel("Etiqueta Real", fontsize=14)
plt.show()

# Extra: Mostrar algunos ejemplos donde el modelo haya acertado/fallado
print("\nGenerando ejemplos visuales de las predicciones...")
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    # Cogemos índices al azar
    idx = np.random.randint(0, len(etiquetas_reales))
    img = X_test[idx].squeeze().numpy()
    real = etiquetas_reales[idx]
    pred = predicciones_totales[idx]
    
    axes[i].imshow(img, cmap='gray')
    # Pintamos el título en verde si acertó, en rojo si falló
    color = 'green' if real == pred else 'red'
    axes[i].set_title(f"Real: {real:02d} | Pred: {pred:02d}", color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()